# Applied Data Mining with Python

## Part 1: Foundations & Data Understanding

*dr. Uroš Godnov*  ·  Notebook companion to the Quarto slide deck.

> This notebook runs top-to-bottom on Google Colab's free tier. Each **Hands-on Exercise** has a `# TODO` cell to complete, followed by a collapsible **💡 Show solution**. Run the setup cells in order first.

In [ ]:
# --- Robust data loader (added for the notebook version) -------------------------------
# Loads the wine CSV from the UCI repository, falling back to a GitHub mirror if UCI is
# unreachable during a live session. Used everywhere the deck wrote pd.read_csv(url, ...).
import pandas as pd

def load_wine(url, sep=";"):
    try:
        return pd.read_csv(url, sep=sep)
    except Exception as e:
        fname = url.rsplit("/", 1)[-1]
        mirror = "https://raw.githubusercontent.com/zygmuntz/wine-quality/master/winequality/" + fname
        print(f"Primary source failed ({type(e).__name__}); falling back to mirror.")
        return pd.read_csv(mirror, sep=sep)

## Course Overview

**Part 1: Foundations & Data Understanding** (≈ 3 hours)

| Module | Topic | Duration |
|--------|-------|----------|
| 1.1 | Data Collection and Loading | 30 min |
| 1.2 | Exploratory Data Analysis (EDA) | 45 min |
| 1.3 | Intermediate Pandas Operations | 45 min |
| 1.4 | Data Cleaning and Preprocessing | 45 min |
| 1.5 | Wow Moment: Insights from Raw Data | 15 min |

## Learning Objectives

By the end of this session, you will be able to:

- Load data from multiple sources into pandas DataFrames
- Conduct systematic exploratory data analysis
- Apply intermediate pandas operations for data manipulation
- Identify and handle missing values, outliers, and data quality issues
- Generate compelling visualizations that reveal hidden patterns

# Module 1.1: Data Collection and Loading

## Conceptual Overview

Data acquisition is the foundation of any data mining project. Real-world data comes in various formats and from multiple sources. Mastering data loading ensures you can work with any dataset you encounter.

Common data sources include:

- CSV and Excel files (local and remote)
- APIs and web scraping results
- Databases (SQL queries)
- Built-in library datasets for learning

## Wow Factor Hook

You will load a real-world dataset about wine quality and within minutes understand its structure, detect potential issues, and identify which columns might predict quality scores.

## Dataset Introduction

We will use the **Wine Quality Dataset** from the UCI Machine Learning Repository. This dataset contains physicochemical properties of Portuguese wines and quality ratings from expert tasters.

Why it matters: Wine quality prediction has real commercial value, and the features are intuitive enough to build domain understanding quickly.

## Code: Loading Data from URLs

In [ ]:
import pandas as pd
import numpy as np

# Set display options for better readability
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 200)

# Load wine quality dataset directly from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_df = load_wine(url)

# First look at the data
print(f"Dataset shape: {wine_df.shape}")
print(f"\nColumn names:\n{wine_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
wine_df.head()

## Code: Loading Built-in Datasets

In [ ]:
from sklearn.datasets import load_iris, fetch_california_housing

# Load California Housing dataset (sklearn built-in)
housing = fetch_california_housing(as_frame=True)
housing_df = housing.frame

print(f"Housing dataset shape: {housing_df.shape}")
print(f"\nFeature names: {housing.feature_names}")
print(f"\nTarget: {housing.target_names}")
housing_df.head()

## Code: Basic Data Inspection

In [ ]:
# Comprehensive first inspection
print("=" * 50)
print("DATA TYPES")
print("=" * 50)
print(wine_df.dtypes)

print("\n" + "=" * 50)
print("BASIC STATISTICS")
print("=" * 50)
print(wine_df.describe())

print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)
print(wine_df.isnull().sum())

print("\n" + "=" * 50)
print("MEMORY USAGE")
print("=" * 50)
print(wine_df.info())

## Expected Output & Interpretation

You should observe:

- **Shape**: 1,599 rows × 12 columns (manageable size for learning)
- **Data types**: All numeric (float64 and int64) — no immediate type conversion needed
- **No missing values**: Clean dataset ready for analysis
- **Quality column**: Integer ratings from 3-8 (our target variable)

This inspection reveals a well-structured dataset ideal for regression or classification tasks.

## Hands-on Exercise 1.1

**Task**: Load the white wine quality dataset and compare its structure to the red wine dataset.

1. Load the white wine data from: `https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv`
2. Print the shape of both datasets
3. Check if both datasets have the same columns
4. Compare the distribution of quality ratings between red and white wines using `value_counts()`

**Time**: 10 minutes

In [ ]:
# TODO: load the WHITE wine CSV and compare it to the red one
# Hint: white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
#       use load_wine(white_url); compare .shape, columns, and quality value_counts()


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
white_df = load_wine(white_url)

print("Red shape:  ", wine_df.shape)
print("White shape:", white_df.shape)
print("Same columns:", list(wine_df.columns) == list(white_df.columns))

print("\nRed quality counts:")
print(wine_df["quality"].value_counts().sort_index())
print("\nWhite quality counts:")
print(white_df["quality"].value_counts().sort_index())
```

</details>

# Module 1.2: Exploratory Data Analysis (EDA)

## Conceptual Overview

EDA is the systematic process of investigating data before formal modeling. It helps you:

- Understand variable distributions
- Identify relationships between features
- Detect anomalies and outliers
- Form hypotheses about predictive power

A thorough EDA prevents modeling mistakes and reveals opportunities for feature engineering.

## Wow Factor Hook

You will create a comprehensive visual analysis that reveals which chemical properties most strongly influence wine quality — all before building any model.

## Code: Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for all visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Create distribution plots for all numeric features
fig, axes = plt.subplots(4, 3, figsize=(15, 12))
axes = axes.flatten()

for idx, column in enumerate(wine_df.columns):
    ax = axes[idx]
    wine_df[column].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(column, fontsize=10)
    ax.set_xlabel('')
    
plt.tight_layout()
plt.suptitle('Distribution of Wine Quality Features', y=1.02, fontsize=14)
plt.show()

## Code: Target Variable Analysis

In [ ]:
# Analyze the target variable: quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
quality_counts = wine_df['quality'].value_counts().sort_index()
axes[0].bar(quality_counts.index, quality_counts.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Quality Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Wine Quality Ratings')

# Percentage breakdown
quality_pct = (wine_df['quality'].value_counts(normalize=True) * 100).sort_index()
axes[1].pie(quality_pct.values, labels=quality_pct.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Quality Rating Proportions')

plt.tight_layout()
plt.show()

print("\nQuality distribution:")
print(wine_df['quality'].value_counts().sort_index())

## Code: Correlation Analysis

In [ ]:
# Compute correlation matrix
correlation_matrix = wine_df.corr(numeric_only=True)

# Create heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, 
            mask=mask,
            annot=True, 
            fmt='.2f', 
            cmap='RdBu_r',
            center=0,
            square=True,
            linewidths=0.5)
plt.title('Correlation Matrix: Wine Quality Features', fontsize=14)
plt.tight_layout()
plt.show()

# Extract correlations with target
print("\nCorrelations with Quality (sorted by absolute value):")
quality_corr = correlation_matrix['quality'].drop('quality').abs().sort_values(ascending=False)
print(quality_corr)

## Code: Feature-Target Relationships

In [ ]:
# Box plots showing feature distributions across quality levels
top_features = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    wine_df.boxplot(column=feature, by='quality', ax=ax)
    ax.set_xlabel('Quality Rating')
    ax.set_ylabel(feature)
    ax.set_title(f'{feature} by Quality')
    
plt.suptitle('Top Correlated Features vs Wine Quality', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Expected Output & Interpretation

Key findings from EDA:

- **Quality distribution**: Most wines rated 5-6, few at extremes (class imbalance)
- **Strongest positive correlations**: Alcohol (0.48), sulphates (0.25), citric acid (0.23)
- **Strongest negative correlations**: Volatile acidity (-0.39)
- **Visual insight**: Higher alcohol content clearly associated with higher quality ratings

These findings will guide feature engineering and model interpretation.

## Hands-on Exercise 1.2

**Task**: Perform EDA on the relationship between pH and quality.

1. Create a scatter plot of pH vs quality (use jitter or transparency for overlapping points)
2. Calculate the mean pH for each quality level
3. Create a box plot of pH grouped by quality
4. Write a one-sentence interpretation of what you observe

**Time**: 15 minutes

In [ ]:
# TODO: explore pH vs quality (scatter with jitter, mean pH per quality, boxplot, 1-sentence read)
# Hint: jitter = np.random.normal(0, 0.08, len(wine_df)); group with wine_df.groupby('quality')['pH'].mean()


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
import matplotlib.pyplot as plt

# 1. scatter with jitter so overlapping integer quality values are visible
jitter = np.random.normal(0, 0.08, len(wine_df))
plt.figure(figsize=(8, 5))
plt.scatter(wine_df["quality"] + jitter, wine_df["pH"], alpha=0.3, s=12)
plt.xlabel("Quality"); plt.ylabel("pH"); plt.title("pH vs Quality (jittered)")
plt.show()

# 2. mean pH per quality level
print(wine_df.groupby("quality")["pH"].mean().round(3))

# 3. boxplot of pH by quality
wine_df.boxplot(column="pH", by="quality", figsize=(8, 5))
plt.suptitle(""); plt.title("pH by Quality"); plt.show()

# 4. Interpretation
print("pH barely shifts across quality levels -> it is a weak predictor of quality.")
```

</details>

# Module 1.3: Intermediate Pandas Operations

## Conceptual Overview

Beyond basic selection and filtering, pandas offers powerful operations for data transformation:

- Groupby aggregations for summary statistics
- Apply functions for custom transformations
- Merge and join operations for combining datasets
- Reshaping with pivot tables and melt

These operations are essential for feature engineering and data preparation.

## Wow Factor Hook

You will transform raw wine data into aggregated insights, create derived features, and combine multiple datasets — operations that form the backbone of professional data work.

## Code: GroupBy Operations

In [ ]:
# Group by quality and compute multiple statistics
quality_summary = wine_df.groupby('quality').agg({
    'alcohol': ['mean', 'std', 'min', 'max'],
    'volatile acidity': ['mean', 'std'],
    'sulphates': ['mean', 'std'],
    'pH': ['mean', 'std']
}).round(3)

print("Summary Statistics by Quality Level:")
print(quality_summary)

# Simpler aggregation with named columns
quality_means = wine_df.groupby('quality').agg(
    avg_alcohol=('alcohol', 'mean'),
    avg_volatility=('volatile acidity', 'mean'),
    count=('alcohol', 'count')
).round(3)

print("\nMean values by quality:")
print(quality_means)

## Code: Creating Derived Features

In [ ]:
# Create a working copy
wine_features = wine_df.copy()

# Ratio features
wine_features['free_sulfur_ratio'] = (
    wine_features['free sulfur dioxide'] / wine_features['total sulfur dioxide']
)

# Categorical binning
wine_features['alcohol_category'] = pd.cut(
    wine_features['alcohol'],
    bins=[0, 10, 12, 15],
    labels=['low', 'medium', 'high']
)

# Binary classification target
wine_features['is_good'] = (wine_features['quality'] >= 7).astype(int)

# Interaction features
wine_features['acid_alcohol_interaction'] = (
    wine_features['volatile acidity'] * wine_features['alcohol']
)

print("New features created:")
print(wine_features[['free_sulfur_ratio', 'alcohol_category', 'is_good', 
                      'acid_alcohol_interaction']].head(10))

## Code: Apply and Transform

In [ ]:
# Custom function with apply
def categorize_acidity(ph):
    """Categorize wine based on pH level."""
    if ph < 3.0:
        return 'high_acid'
    elif ph < 3.3:
        return 'medium_acid'
    else:
        return 'low_acid'

wine_features['acidity_category'] = wine_features['pH'].apply(categorize_acidity)
print("Acidity categories:")
print(wine_features['acidity_category'].value_counts())

# Z-score normalization using transform
def z_score(x):
    return (x - x.mean()) / x.std()

wine_features['alcohol_zscore'] = wine_features.groupby('quality')['alcohol'].transform(z_score)
print("\nAlcohol Z-scores within quality groups:")
print(wine_features[['quality', 'alcohol', 'alcohol_zscore']].head(10))

## Code: Combining DataFrames

In [ ]:
# Load white wine for combination
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
white_df = load_wine(white_url)

# Add wine type identifier
red_df = wine_df.copy()
red_df['wine_type'] = 'red'
white_df['wine_type'] = 'white'

# Concatenate datasets
combined_wine = pd.concat([red_df, white_df], ignore_index=True)
print(f"Combined dataset shape: {combined_wine.shape}")
print(f"\nWine type counts:\n{combined_wine['wine_type'].value_counts()}")

# Compare means between red and white
comparison = combined_wine.groupby('wine_type')[['alcohol', 'pH', 'quality']].mean()
print(f"\nMean comparison:\n{comparison}")

## Expected Output & Interpretation

These operations reveal:

- **GroupBy insights**: Clear patterns emerge when aggregating by quality level
- **Derived features**: Ratios and categories can capture non-linear relationships
- **Combined data**: White wines outnumber red wines 3:1 in this dataset
- **Type differences**: White wines have slightly higher average quality but lower alcohol

## Hands-on Exercise 1.3

**Task**: Create advanced aggregations and derived features.

1. Create a new feature: `total_acidity = fixed acidity + volatile acidity + citric acid`
2. Group by `quality` and calculate the 25th, 50th, and 75th percentile of `total_acidity`
3. Create a binary feature `high_alcohol` that is 1 when alcohol > 11, otherwise 0
4. Calculate the percentage of `high_alcohol` wines in each quality category

**Time**: 15 minutes

In [ ]:
# TODO: build total_acidity, its quartiles per quality, a high_alcohol flag, and its share per quality
# Hint: pd.cut not needed; (wine_df['alcohol'] > 11).astype(int); groupby('quality')[...].quantile([.25,.5,.75])


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
wine_ex = wine_df.copy()

# 1. total acidity
wine_ex["total_acidity"] = (
    wine_ex["fixed acidity"] + wine_ex["volatile acidity"] + wine_ex["citric acid"]
)

# 2. quartiles of total_acidity per quality
print(
    wine_ex.groupby("quality")["total_acidity"]
    .quantile([0.25, 0.5, 0.75]).unstack().round(3)
)

# 3. high_alcohol flag
wine_ex["high_alcohol"] = (wine_ex["alcohol"] > 11).astype(int)

# 4. share of high-alcohol wines per quality level
print((wine_ex.groupby("quality")["high_alcohol"].mean() * 100).round(1))
```

</details>

# Module 1.4: Data Cleaning and Preprocessing

## Conceptual Overview

Real-world data is rarely clean. Effective preprocessing addresses:

- Missing values (imputation or removal)
- Outliers (detection and treatment)
- Inconsistent formats and duplicates
- Feature scaling for modeling

Clean data is the prerequisite for reliable models.

## Wow Factor Hook

You will systematically detect and handle data quality issues, transforming messy data into model-ready features through a reproducible pipeline.

## Dataset Introduction

We will introduce artificial data quality issues into the wine dataset to practice handling real-world problems, then work with the California Housing dataset which has natural complexities.

## Code: Introducing and Detecting Missing Values

In [ ]:
import numpy as np

# Create a copy with artificial missing values
np.random.seed(42)
wine_dirty = wine_df.copy()

# Introduce missing values randomly (5% of each column)
for col in wine_dirty.columns:
    mask = np.random.random(len(wine_dirty)) < 0.05
    wine_dirty.loc[mask, col] = np.nan

# Detect missing values
print("Missing values per column:")
missing_report = pd.DataFrame({
    'missing_count': wine_dirty.isnull().sum(),
    'missing_pct': (wine_dirty.isnull().sum() / len(wine_dirty) * 100).round(2)
})
print(missing_report[missing_report['missing_count'] > 0])

# Visualize missing pattern
plt.figure(figsize=(12, 6))
sns.heatmap(wine_dirty.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Value Pattern')
plt.tight_layout()
plt.show()

## Code: Handling Missing Values

In [ ]:
# Strategy 1: Drop rows with any missing values
wine_dropped = wine_dirty.dropna()
print(f"After dropping: {len(wine_dropped)} rows (lost {len(wine_dirty) - len(wine_dropped)})")

# Strategy 2: Impute with mean
wine_mean_imputed = wine_dirty.copy()
for col in wine_mean_imputed.columns:
    wine_mean_imputed[col] = wine_mean_imputed[col].fillna(wine_mean_imputed[col].mean())

# Strategy 3: Impute with median (more robust to outliers)
wine_median_imputed = wine_dirty.copy()
for col in wine_median_imputed.columns:
    wine_median_imputed[col] = wine_median_imputed[col].fillna(wine_median_imputed[col].median())

# Strategy 4: Using sklearn imputer
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
wine_sklearn_imputed = pd.DataFrame(
    imputer.fit_transform(wine_dirty),
    columns=wine_dirty.columns
)

print(f"\nMissing values after sklearn imputation: {wine_sklearn_imputed.isnull().sum().sum()}")

## Code: Outlier Detection

In [ ]:
def detect_outliers_iqr(df, column, threshold=1.5):
    """Detect outliers using IQR method."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers in key features
print("Outlier Detection Report (IQR method):")
print("-" * 50)
for col in ['residual sugar', 'chlorides', 'total sulfur dioxide']:
    outliers, lower, upper = detect_outliers_iqr(wine_df, col)
    print(f"\n{col}:")
    print(f"  Bounds: [{lower:.3f}, {upper:.3f}]")
    print(f"  Outliers: {len(outliers)} ({len(outliers)/len(wine_df)*100:.1f}%)")

# Visualize outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, col in enumerate(['residual sugar', 'chlorides', 'total sulfur dioxide']):
    axes[idx].boxplot(wine_df[col])
    axes[idx].set_title(f'{col}')
    axes[idx].set_ylabel('Value')
plt.tight_layout()
plt.show()

## Code: Handling Outliers

In [ ]:
def cap_outliers(df, column, threshold=1.5):
    """Cap outliers at IQR bounds."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return df

# Create capped version
wine_capped = wine_df.copy()
for col in ['residual sugar', 'chlorides', 'total sulfur dioxide']:
    wine_capped = cap_outliers(wine_capped, col)

# Compare distributions before and after
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols_to_check = ['residual sugar', 'chlorides', 'total sulfur dioxide']

for idx, col in enumerate(cols_to_check):
    # Original
    axes[0, idx].hist(wine_df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[0, idx].set_title(f'{col} (Original)')
    # Capped
    axes[1, idx].hist(wine_capped[col], bins=30, edgecolor='black', alpha=0.7, color='green')
    axes[1, idx].set_title(f'{col} (Capped)')

plt.tight_layout()
plt.show()

## Code: Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Select numeric features (excluding target)
features = wine_df.drop('quality', axis=1)

# Standard Scaling (z-score)
scaler_standard = StandardScaler()
features_standard = pd.DataFrame(
    scaler_standard.fit_transform(features),
    columns=features.columns
)

# Min-Max Scaling (0-1 range)
scaler_minmax = MinMaxScaler()
features_minmax = pd.DataFrame(
    scaler_minmax.fit_transform(features),
    columns=features.columns
)

# Robust Scaling (using median and IQR)
scaler_robust = RobustScaler()
features_robust = pd.DataFrame(
    scaler_robust.fit_transform(features),
    columns=features.columns
)

# Compare scaling effects
print("Original statistics:")
print(features[['alcohol', 'pH', 'chlorides']].describe().round(3))
print("\nStandard scaled:")
print(features_standard[['alcohol', 'pH', 'chlorides']].describe().round(3))
print("\nMin-Max scaled:")
print(features_minmax[['alcohol', 'pH', 'chlorides']].describe().round(3))

## Expected Output & Interpretation

Preprocessing accomplishments:

- **Missing values**: Multiple imputation strategies available; median imputation is robust to outliers
- **Outliers**: IQR method identifies 5-10% outliers in some features; capping preserves data while reducing extreme influence
- **Scaling**: StandardScaler centers at 0 with unit variance; essential for algorithms sensitive to feature magnitude

## Hands-on Exercise 1.4

**Task**: Build a complete preprocessing pipeline.

1. Load the California Housing dataset from sklearn
2. Check for missing values (there should be none, but verify)
3. Detect outliers in `MedInc` (median income) using the IQR method
4. Apply StandardScaler to all features
5. Print the mean and standard deviation of scaled features to verify (should be ~0 and ~1)

**Time**: 15 minutes

In [ ]:
# TODO: preprocessing pipeline on California Housing (missing check, MedInc IQR outliers, StandardScaler)
# Hint: from sklearn.datasets import fetch_california_housing; df = fetch_california_housing(as_frame=True).frame


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

cal = fetch_california_housing(as_frame=True)
hdf = cal.frame

# 2. verify no missing values
print("Total missing values:", hdf.isnull().sum().sum())

# 3. IQR outliers in MedInc
q1, q3 = hdf["MedInc"].quantile([0.25, 0.75]); iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = hdf[(hdf["MedInc"] < low) | (hdf["MedInc"] > high)]
print(f"MedInc outliers: {len(out)} ({len(out) / len(hdf) * 100:.1f}%)")

# 4-5. scale all features (target MedHouseVal excluded) and verify mean~0, std~1
feat = hdf.drop(columns=["MedHouseVal"])
scaled = pd.DataFrame(StandardScaler().fit_transform(feat), columns=feat.columns)
print(scaled.describe().loc[["mean", "std"]].round(3))
```

</details>

# Module 1.5: Wow Moment — Insights from Raw Data

## Wow Factor: Comprehensive EDA Dashboard

Now we combine everything into a compelling analysis that reveals actionable insights about what makes a high-quality wine.

## Code: Executive Summary Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create comprehensive dashboard
fig = plt.figure(figsize=(18, 12))

# 1. Quality distribution
ax1 = fig.add_subplot(2, 3, 1)
wine_df['quality'].value_counts().sort_index().plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
ax1.set_title('Quality Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Quality Rating')
ax1.set_ylabel('Count')

# 2. Top correlations with quality
ax2 = fig.add_subplot(2, 3, 2)
quality_corr = wine_df.corr(numeric_only=True)['quality'].drop('quality').sort_values()
quality_corr.plot(kind='barh', ax=ax2, color=['crimson' if x < 0 else 'forestgreen' for x in quality_corr])
ax2.set_title('Feature Correlations with Quality', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# 3. Alcohol vs Quality scatter with trend
ax3 = fig.add_subplot(2, 3, 3)
quality_alcohol = wine_df.groupby('quality')['alcohol'].mean()
ax3.scatter(wine_df['quality'] + np.random.normal(0, 0.1, len(wine_df)), 
            wine_df['alcohol'], alpha=0.3, s=10)
ax3.plot(quality_alcohol.index, quality_alcohol.values, 'r-', linewidth=3, label='Mean')
ax3.set_title('Alcohol Content vs Quality', fontsize=12, fontweight='bold')
ax3.set_xlabel('Quality')
ax3.set_ylabel('Alcohol %')
ax3.legend()

# 4. High vs Low quality comparison
ax4 = fig.add_subplot(2, 3, 4)
wine_df['quality_group'] = wine_df['quality'].apply(lambda x: 'High (7-8)' if x >= 7 else ('Low (3-4)' if x <= 4 else 'Medium (5-6)'))
features_to_compare = ['alcohol', 'volatile acidity', 'sulphates']
comparison_data = wine_df.groupby('quality_group')[features_to_compare].mean()
comparison_data.plot(kind='bar', ax=ax4, edgecolor='black')
ax4.set_title('Key Features by Quality Group', fontsize=12, fontweight='bold')
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
ax4.legend(loc='upper right')

# 5. Volatile acidity distribution by quality
ax5 = fig.add_subplot(2, 3, 5)
for quality in sorted(wine_df['quality'].unique()):
    subset = wine_df[wine_df['quality'] == quality]['volatile acidity']
    ax5.hist(subset, alpha=0.5, label=f'Q={quality}', bins=20)
ax5.set_title('Volatile Acidity by Quality', fontsize=12, fontweight='bold')
ax5.set_xlabel('Volatile Acidity')
ax5.legend()

# 6. Summary statistics table
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
summary_stats = wine_df.groupby('quality')[['alcohol', 'volatile acidity', 'sulphates']].mean().round(2)
table = ax6.table(cellText=summary_stats.values,
                   rowLabels=summary_stats.index,
                   colLabels=summary_stats.columns,
                   loc='center',
                   cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax6.set_title('Mean Feature Values by Quality', fontsize=12, fontweight='bold', y=0.9)

plt.tight_layout()
plt.savefig('wine_quality_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up temporary column
wine_df.drop('quality_group', axis=1, inplace=True)

## Code: Key Findings Summary

In [ ]:
print("=" * 60)
print("WINE QUALITY ANALYSIS: KEY FINDINGS")
print("=" * 60)

# Finding 1: Alcohol correlation
high_quality = wine_df[wine_df['quality'] >= 7]['alcohol'].mean()
low_quality = wine_df[wine_df['quality'] <= 4]['alcohol'].mean()
print(f"\n1. ALCOHOL CONTENT")
print(f"   High quality wines (≥7): {high_quality:.2f}% average alcohol")
print(f"   Low quality wines (≤4):  {low_quality:.2f}% average alcohol")
print(f"   Difference: +{high_quality - low_quality:.2f}%")

# Finding 2: Volatile acidity
high_va = wine_df[wine_df['quality'] >= 7]['volatile acidity'].mean()
low_va = wine_df[wine_df['quality'] <= 4]['volatile acidity'].mean()
print(f"\n2. VOLATILE ACIDITY (negative indicator)")
print(f"   High quality wines: {high_va:.3f} g/L")
print(f"   Low quality wines:  {low_va:.3f} g/L")
print(f"   High quality wines have {((low_va-high_va)/low_va*100):.0f}% lower volatile acidity")

# Finding 3: Prediction potential
top_3_features = wine_df.corr(numeric_only=True)['quality'].drop('quality').abs().nlargest(3)
print(f"\n3. TOP PREDICTIVE FEATURES:")
for feature, corr in top_3_features.items():
    print(f"   - {feature}: |r| = {corr:.3f}")

# Finding 4: Class imbalance
print(f"\n4. CLASS DISTRIBUTION CHALLENGE:")
mid_quality = len(wine_df[wine_df['quality'].isin([5, 6])]) / len(wine_df) * 100
print(f"   {mid_quality:.1f}% of wines are rated 5 or 6")
print(f"   This imbalance will affect model training")

print("\n" + "=" * 60)
print("READY FOR MODELING")
print("=" * 60)

## Conclusion: Part 1

You have completed the first phase of the data mining lifecycle:

| Skill | Accomplishment |
|-------|----------------|
| Data Loading | Loaded data from URLs and sklearn |
| EDA | Created comprehensive visualizations |
| Pandas Operations | Applied groupby, apply, and merge |
| Preprocessing | Handled missing values, outliers, and scaling |
| Insights | Identified key predictive features |

Next session: **Feature Engineering & Classical Models**

## Part 1 Wrap-up Exercise

**Comprehensive Task**: Apply everything you learned to the California Housing dataset.

1. Load the dataset and perform complete data inspection
2. Create at least 3 visualizations exploring feature-target relationships
3. Identify the top 3 features correlated with house value
4. Handle any outliers using the capping method
5. Scale all features using StandardScaler
6. Write a brief (3-5 sentence) summary of your findings

**Time**: 30 minutes (homework if needed)

In [ ]:
# TODO: full EDA + preprocessing on California Housing (inspect, 3 plots, top-3 corr, cap outliers, scale, summary)
# Hint: reuse the IQR capping and StandardScaler patterns from this part


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df = fetch_california_housing(as_frame=True).frame.copy()

# 1. inspection
print(df.describe().round(2))

# 3. top-3 features correlated with house value
corr = (df.corr(numeric_only=True)["MedHouseVal"].drop("MedHouseVal")
          .sort_values(key=abs, ascending=False))
print("\nTop 3 correlated with MedHouseVal:")
print(corr.head(3).round(3))

# 2. three feature-target visualizations
top3 = corr.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, top3):
    ax.scatter(df[f], df["MedHouseVal"], alpha=0.2, s=8)
    ax.set_xlabel(f); ax.set_ylabel("MedHouseVal")
plt.tight_layout(); plt.show()

# 4. cap outliers in MedInc
def cap(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return s.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
df["MedInc"] = cap(df["MedInc"])

# 5. scale features
feat = df.drop(columns=["MedHouseVal"])
scaled = pd.DataFrame(StandardScaler().fit_transform(feat), columns=feat.columns)
print("\nScaled feature means (~0):")
print(scaled.mean().round(3))

# 6. summary
print("\nMedInc (income) is the dominant driver of house value; location "
      "(Latitude/Longitude) matters too. After capping income outliers and "
      "standardizing, the features are model-ready.")
```

</details>

## References and Resources

- Pandas Documentation: https://pandas.pydata.org/docs/
- Seaborn Gallery: https://seaborn.pydata.org/examples/
- Scikit-learn Preprocessing: https://scikit-learn.org/stable/modules/preprocessing.html
- UCI Wine Quality Dataset: https://archive.ics.uci.edu/ml/datasets/wine+quality

## End of Part 1

Continue to **Part 2: Feature Engineering & Classical Models**